# Distinct Rows — Removing Duplicate Rows

Remove duplicate records across the intermediate DataFrame with full control over column subsets.

`pytae` provides both the method-chaining accessor `df.pt.distinct(*cols, keep='first')` and the functional verb `pt.distinct(df, *cols)`. In the CLI, the equivalent flag is `-distinct` (or `-distinct COLS`).

| Operation | Pytae Method Chain | CLI Equivalent |
|---|---|---|
| All Columns | `df.pt.distinct()` | `-distinct` |
| Specific Columns | `df.pt.distinct('col1', 'col2')` | `-distinct "col1,col2"` |
| Keep Last / None | `df.pt.distinct(keep='last')` | N/A |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import pandas as pd

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']

## 1. Deduplicating Across All Columns (`.pt.distinct()`)

Eliminate identical rows across all selected columns. The index is automatically reset:

In [2]:
# Keep only unique species and island pairs
(
    penguins
    .pt.select("species", "island")
    .pt.distinct()
)

,species,island
0,Adelie,Torgersen
1,Adelie,Biscoe
2,Adelie,Dream
3,Chinstrap,Dream
4,Gentoo,Biscoe


## 2. Deduplicating by Specific Identifier Columns

Pass column names to restrict duplicate checks to specific identifiers, keeping the first matching record for all other columns:

In [3]:
# Keep the first recorded penguin for each unique species and island combination
(
    penguins
    .pt.select("species", "island", "body_mass_g", "sex")
    .pt.distinct("species", "island")
)

,species,island,body_mass_g,sex
0,Adelie,Torgersen,3750.0,Male
1,Adelie,Biscoe,3400.0,Female
2,Adelie,Dream,3250.0,Female
3,Chinstrap,Dream,3500.0,Female
4,Gentoo,Biscoe,4500.0,Female


## 3. Chaining in Pipelines

`df.pt.distinct()` flows seamlessly inside multi-step pipelines without dropping out of `.pt`:

In [4]:
# Filter heavy penguins -> select columns -> deduplicate -> arrange
(
    penguins
    .pt.qry("body_mass_g > 4000")
    .pt.select("species", "island")
    .pt.distinct()
    .pt.arrange("species", "island")
)

,species,island
0,Adelie,Biscoe
1,Adelie,Dream
2,Adelie,Torgersen
3,Chinstrap,Dream
4,Gentoo,Biscoe


## 4. Keeping the Last Record (`keep='last'`)

Optionally retain the last occurrence instead of the first:

In [5]:
# Retain the last record for each day and time
(
    tips
    .pt.select("day", "time", "total_bill", "tip")
    .pt.distinct("day", "time", keep="last")
)

,day,time,total_bill,tip
0,Fri,Dinner,15.38,3.00
1,Sun,Dinner,15.69,1.50
2,Thur,Lunch,16.47,3.23
3,Fri,Lunch,10.09,2.00
4,Sat,Dinner,17.82,1.75
5,Thur,Dinner,18.78,3.00


## 5. Functional Style (`pt.distinct`)

Available as a standalone function:

In [6]:
# Functional deduplication
pt.distinct(penguins[["species", "island"]])

,species,island
0,Adelie,Torgersen
1,Adelie,Biscoe
2,Adelie,Dream
3,Chinstrap,Dream
4,Gentoo,Biscoe
